# Train and validate ActionFormer every epochs

## Import ActionFormer lib

In [2]:
# python imports
import argparse
import os
import time
import datetime
from pprint import pprint

# torch imports
import torch
import torch.nn as nn
import torch.utils.data
# for visualization
from torch.utils.tensorboard import SummaryWriter

# our code
from actionformer2.libs.core import load_config
from actionformer2.libs.datasets import make_dataset, make_data_loader
from actionformer2.libs.modeling import make_meta_arch
from actionformer2.libs.utils import (train_one_epoch, valid_one_epoch, valid_one_epoch_multi, ANETdetection,
                        save_checkpoint, make_optimizer, make_scheduler,
                        fix_random_seed, ModelEma)

## Args

In [3]:
config = "configs/finebio_vjepa2_multi.yaml"
output = "test"
ckpt_file = "outputs/multi/vjepa2/finebio_vjepa2_multi_test/not_exist.pth.tar"
apply_graph_modeling = False
pred_op = True
op_pred_method = 'fuse' # fuse or cls
resume = False
print_freq = 10
ckpt_freq = 5
pivot_type = None
saveonly = False
stat_dir = None
val_freq = 10

# Training and validation loop

In [4]:
if os.path.exists(config):
    cfg = load_config(config)
else:
    raise ValueError("Config file does not exist.")

# prep for output folder (based on time stamp)
if not os.path.exists(cfg['output_folder']):
    os.mkdir(cfg['output_folder'])
cfg_filename = os.path.basename(config).replace('.yaml', '')
if len(output) == 0:
    ts = datetime.datetime.fromtimestamp(int(time.time()))
    ckpt_folder = os.path.join(
        cfg['output_folder'], cfg_filename + '_' + str(ts))
else:
    ckpt_folder = os.path.join(
        cfg['output_folder'], cfg_filename + '_' + str(output))
if not os.path.exists(ckpt_folder):
    os.mkdir(ckpt_folder)
# tensorboard writer
tb_writer = SummaryWriter(os.path.join(ckpt_folder, 'logs'))

# fix the random seeds (this will fix everything)
rng_generator = fix_random_seed(cfg['init_rand_seed'], include_cuda=True)

# re-scale learning rate / # workers based on number of GPUs
cfg['opt']["learning_rate"] *= len(cfg['devices'])
cfg['loader']['num_workers'] *= len(cfg['devices'])

"""2. create dataset / dataloader"""
## train dataset =====
train_dataset = make_dataset(
    cfg['dataset_name'], True, cfg['train_split'], **cfg['dataset']
)
# update cfg based on dataset attributes (fix to epic-kitchens)
train_db_vars = train_dataset.get_attributes()
cfg['model']['train_cfg']['head_empty_cls'] = train_db_vars['empty_label_ids']
# num_classes for all types to train
cfg["model"]["num_classes"] = train_dataset.num_classes[0] if len(train_dataset.num_classes) == 1 else train_dataset.num_classes
if cfg["model_name"] == "LocPointTransformer":
    cfg["model"]["apply_graph_modeling"] = apply_graph_modeling
elif cfg["model_name"] == "MultiPredictionLocPointTransformer":
    if pred_op:
        assert len(set(train_dataset.original_type_names) - set(["verb", "manipulated", "affected"])) == 0, \
            "Model should have three heads (verb/manipulated/affected) to predict operations based on entities."
    if not pred_op and apply_graph_modeling:
        print("WARNING: Operation graph modeling is enabled only when operation is predicted. Please add an argument '--pred_op'.")
    cfg["model"]["with_op_pred"] = pred_op
    cfg["model"]["op_pred_method"] = op_pred_method
    cfg["model"]["apply_op_graph_modeling"] = apply_graph_modeling

# data loaders
train_loader = make_data_loader(
    train_dataset, True, rng_generator, **cfg['loader'])

## val dataset =====
val_dataset = make_dataset(
    cfg['dataset_name'], False, cfg['val_split'], **cfg['dataset']
)
assert len(val_dataset) > 0, "Validation dataset is empty."
# num_classes for all types to predict in the model.
cfg["model"]["num_classes"] = val_dataset.num_classes.copy()

# when operation is also inferenced from verb, manipulated and affected.
if pred_op:
    assert len(set(val_dataset.original_type_names) - set(["verb", "manipulated", "affected"])) == 0, \
        "Model should have three heads (verb/manipulated/affected) to predict operations based on entities."
    assert len(val_dataset.hands) == 1, "all types should correspond to the same side of hand."
    dataset_config = cfg["dataset"].copy()
    dataset_config["type_names"] = ["atomic_operation"]
    action_dataset = make_dataset(
        cfg['dataset_name'], False, cfg['val_split'], **dataset_config
    )
if not pred_op and apply_graph_modeling:
    print("WARNING: Operation graph modeling is enabled only when operation is predicted. Please add an argument '--pred_op'.")
cfg["model"]["with_op_pred"] = pred_op
cfg["model"]["op_pred_method"] = op_pred_method
cfg["model"]["apply_op_graph_modeling"] = apply_graph_modeling

# set bs = 1, and disable shuffle
val_loader = make_data_loader(
    val_dataset, False, None, 1, cfg['loader']['num_workers']
)
type_names = val_dataset.type_names.copy()
original_type_names = val_dataset.type_names.copy()
num_classes = val_dataset.num_classes.copy()
type_groups_with_same_hand = val_dataset.type_groups_with_same_hand.copy()
if pred_op:
    # update type_names/original_type_names, num_classes
    type_names = type_names + action_dataset.type_names
    original_type_names = original_type_names + action_dataset.original_type_names
    num_classes = num_classes + action_dataset.num_classes
    # add action index in type_groups_with_same_hand
    type_groups_with_same_hand[0] = type_groups_with_same_hand[0] + [3]

# convert format of pivot_type from str to index
pivot_types = []
if pivot_type:
    for i, original_type_name in enumerate(original_type_names):
        if pivot_type == original_type_name:
            pivot_types.append(i)
    assert len(pivot_types) == len(type_groups_with_same_hand)
    print(f"Pivot type: {[type_names[i] for i in pivot_types]}")

# set up evaluator
det_evals, output_files = [], []
val_db_vars = val_dataset.get_attributes()
os.makedirs(os.path.join(os.path.split(ckpt_file)[-2], 'results'), exist_ok=True)
if not saveonly:
    det_evals = []
    for i, type_name in enumerate(val_dataset.type_names):
        det_evals.append(ANETdetection(
            val_dataset.data_lists[i],
            val_dataset.split[0],
            tiou_thresholds=val_db_vars['tiou_thresholds'],
            dataset_name=type_name,
            label_dict=val_dataset.label_dicts[i],
            stat_dir=stat_dir
        ))
    if pred_op:
        for i, type_name in enumerate(action_dataset.type_names):
            det_evals.append(ANETdetection(
                action_dataset.data_lists[i],
                action_dataset.split[0],
                tiou_thresholds=val_db_vars['tiou_thresholds'],
                dataset_name=type_name,
                label_dict=action_dataset.label_dicts[i],
                stat_dir=stat_dir
            ))
output_files = [os.path.join(os.path.split(ckpt_file)[-2], 'results', f'{type_name}_eval_results.pkl') for type_name in val_dataset.type_names]
if pred_op:
    output_files += [os.path.join(os.path.split(ckpt_file)[-2], 'results', f'{type_name}_eval_results.pkl') for type_name in action_dataset.type_names]


# print final cfg
pprint(cfg)

"""3. create model, optimizer, and scheduler"""
# model
model = make_meta_arch(cfg['model_name'], **cfg['model'])
# not ideal for multi GPU training, ok for now
model = nn.DataParallel(model, device_ids=cfg['devices'])
# optimizer
optimizer = make_optimizer(model, cfg['opt'])
# schedule
num_iters_per_epoch = len(train_loader)
scheduler = make_scheduler(optimizer, cfg['opt'], num_iters_per_epoch)

# enable model EMA
print("Using model EMA ...")
model_ema = ModelEma(model)

"""4. Resume from model / Misc"""
start_epoch = 0
# resume from a checkpoint?
if resume:
    if os.path.isfile(resume):
        # load ckpt, reset epoch / best rmse
        checkpoint = torch.load(resume,
            map_location = lambda storage, loc: storage.cuda(
                cfg['devices'][0]))
        start_epoch = checkpoint['epoch']
        model.load_state_dict(checkpoint['state_dict'])
        model_ema.module.load_state_dict(checkpoint['state_dict_ema'])
        # also load the optimizer / scheduler if necessary
        optimizer.load_state_dict(checkpoint['optimizer'])
        scheduler.load_state_dict(checkpoint['scheduler'])
        print("=> loaded checkpoint '{:s}' (epoch {:d}".format(
            resume, checkpoint['epoch']
        ))
        del checkpoint
    else:
        print("=> no checkpoint found at '{}'".format(resume))
        raise ValueError("No checkpoint found at specified path.")

# save the current config
with open(os.path.join(ckpt_folder, 'config.txt'), 'w') as fid:
    pprint(cfg, stream=fid)
    fid.flush()

"""4. training / validation loop"""
print("\nStart training model {:s} ...".format(cfg['model_name']))

# start training
max_epochs = cfg['opt'].get(
    'early_stop_epochs',
    cfg['opt']['epochs'] + cfg['opt']['warmup_epochs']
)
for epoch in range(start_epoch, max_epochs):
    # train for one epoch
    train_one_epoch(
        train_loader,
        model,
        optimizer,
        scheduler,
        epoch,
        model_ema = model_ema,
        clip_grad_l2norm = cfg['train_cfg']['clip_grad_l2norm'],
        tb_writer=tb_writer,
        print_freq=print_freq
    )

    # save ckpt once in a while
    if (
        ((epoch + 1) == max_epochs) or
        ((ckpt_freq > 0) and ((epoch + 1) % ckpt_freq == 0))
    ):
        save_states = {
            'epoch': epoch + 1,
            'state_dict': model.state_dict(),
            'scheduler': scheduler.state_dict(),
            'optimizer': optimizer.state_dict(),
        }

        save_states['state_dict_ema'] = model_ema.module.state_dict()
        save_checkpoint(
            save_states,
            False,
            file_folder=ckpt_folder,
            file_name='epoch_{:03d}.pth.tar'.format(epoch + 1)
        )
    if epoch % val_freq == 0:
        valid_one_epoch_multi(
            cfg,
            val_loader,
            model,
            epoch,
            type_names,
            num_classes,
            type_groups_with_same_hand,
            pivot_types=pivot_types,
            evaluator=det_evals, 
            output_files=output_files,
            print_freq=print_freq
        )
        
# wrap up
tb_writer.close()
print("All done!")

{'dataset': {'crop_ratio': [0.9, 1.0],
             'default_fps': None,
             'downsample_rate': 1,
             'feat_folder': 'data/vjepa2_feats',
             'feat_stride': 4,
             'file_ext': '.npy',
             'file_prefix': None,
             'hands': [],
             'input_dim': 1024,
             'json_file': 'data/annotations/annotation_all.json',
             'max_seq_len': 3328,
             'num_frames': 16,
             'trunc_thresh': 0.3,
             'type_names': ['verb', 'manipulated', 'affected']},
 'dataset_name': 'finebio',
 'devices': ['cuda:0'],
 'init_rand_seed': 1234567891,
 'loader': {'batch_size': 2, 'num_workers': 4},
 'model': {'apply_op_graph_modeling': False,
           'backbone_arch': [3, 3, 5],
           'backbone_type': 'convTransformer',
           'embd_dim': 512,
           'embd_kernel_size': 3,
           'embd_with_ln': True,
           'fpn_dim': 512,
           'fpn_start_level': 0,
           'fpn_type': 'fpn',
          

In [7]:
model

DataParallel(
  (module): MultiPredictionPtTransformer(
    (backbone): ConvTransformerBackbone(
      (relu): ReLU(inplace=True)
      (embd): ModuleList(
        (0): MaskedConv1D(
          (conv): Conv1d(1024, 512, kernel_size=(3,), stride=(1,), padding=(1,), bias=False)
        )
        (1-2): 2 x MaskedConv1D(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(1,), padding=(1,), bias=False)
        )
      )
      (embd_norm): ModuleList(
        (0-2): 3 x LayerNorm()
      )
      (stem): ModuleList(
        (0-2): 3 x TransformerBlock(
          (ln1): LayerNorm()
          (ln2): LayerNorm()
          (attn): LocalMaskedMHCA(
            (query_conv): MaskedConv1D(
              (conv): Conv1d(512, 512, kernel_size=(3,), stride=(1,), padding=(1,), groups=512, bias=False)
            )
            (query_norm): LayerNorm()
            (key_conv): MaskedConv1D(
              (conv): Conv1d(512, 512, kernel_size=(3,), stride=(1,), padding=(1,), groups=512, bias=False)